# Partioning performance

In [0]:
%python
spark.conf.set("spark.databricks.io.cache.enabled", False)

In [0]:
%python
from spark.sql import functions as F

df = (spark.range(0, 2500) # create a dataframe with 2500 rows
            .select(F.hash('id').alias('id'), # hash the id to get a unique value
            F.rand().alias('value'), # generate a random value
            F.from_unixtime(F.lit(1701692381 + F.col('id'))).alias('time') # generate a time value
            ))

df.write.mode('overwrite').partitionBy('id').saveAsTable('test_table') # write the dataframe to a table partitioned by id (2500 partitions)

# Data Skipping - is ...
# Z-Ordering - is ...
# Liquid Clustering - is ...
# Predictive Optimization - is ...

In [0]:
ANALYZE TABLE mytable COMPUTE STATISTICS FOR ALL COLUMNS; -- 

# Code optimization


### Skew optimization - AQE - automatically break down larger partitions into smaller, simillar sizes
 1. Salting
 2. add random integers as suffixes to the partition keys

# Shuffle - is side effect of wide transformations (joins, group bys, order bys) and technically of some actions (count etc.)
Re-evaluate join strategy:
- Reordering the join
- Dynamically Switching Join Strategies
- Broadcast Hash Join
- Shuffle hash Joins (default for Databricks Photon)
- Sort-Merge Join (default for OS Spark)



In [ ]:
# Shuffle Demo
from pyspark.sql import functions as F
spark.conf.set('spark.databricks.io.cache.enabled, False)

transactions_df = (spark.range(0, 150000000000, 32)
                        .select('id', F.round(F.rand()*1000, 2).alias('amount'),
                        (F.col('id') % 10).alias('country_id'),
                        (F.col('id') % 100).alias('store_id'),
                        )

transactions_df.write.mode('overwrite').saveAsTable('transactions')

stores_df = (spark.range(0, 99)
                        .select('id', F.round(F.rand()*100, 0).alias('employees'),
                        (F.col('id') % 10).alias('country_id'),
                        F.expr('uuid()').alias('name'),
                        )
stores_df.write.mode('overwrite').saveAsTable('stores')

countries = [(0, 'Italy'), (1, 'Canada')]

columns = ['id', 'name']

countries_df = spark.createDataFrame(data=countries, schema=columns)

In [ ]:
spark.conf.set('spark.sql.autoBroadCastJoinThreshold", -1)
spark.conf.set('spark.databricks.adaptive,autoBroadcastJoinThreshold', -1)

joined_df = spark.sql("""
                        SELECT t.id, amount, c.name as country_name, employees, s.name as store_name
                        FROM transactions as t
                            LEFT JOIN stores as s ON t.store_id = s.id
                            LEFT JOIN countries as c ON t.country_id = c.id
""")

joined_df.write.mode('overwrite').saveAsTable('transact_countries')

In [ ]:
# Enable broadcast join (avoids the shuffle)

spark.conf.unset('spark.sql.autoBroadCastJoinThreshold')
spark.conf.unset('spark.databricks.adaptive,autoBroadcastJoinThreshold')


# Spill - term for moving data from RAM to disk and later back. When partition to large to fit in RAM.
To avoid:
 - allocate cluster with more RAM per Core
 - Address data skew
 - manage size of Spark partitions
 - Avoid expensive operations like explode()
 - Reduce amount data preemtively whever possible

## Serialization
- Spark SQL and DataFrame are highly optimized
- All UDFs must be serialized and distributed to each executor
- Don't use UDF in general, use built-in functions
- Use vectorized UDFs

In [ ]:
from pyspark.sql import functions as F

num_of_cores = 4 # replace with dynamically!

@udf("double")
def F_to_cel(f):
    time.sleep(1)
    return (f-32)*(5/9)

celsius_df = spark.table('device_data').
                    repartition(num_of_cores).
                    withColumn('celsius', F_to_cel(F.col('temp_f'))

# SQL UDFs -- do not require serialization!, Catalyst also can work with SQL UDFs

%sql
CREATE OR REPLACE FUNCTION farh_to_cels (farh DOUBLE)
RETURNS DOUBLE RETURN ((farh-32)*(5/9);

SELECT farh_to_cels(temp_f) FROM device_data;

# Fine Tuning: Choosing the Right Cluster

### Cluster types:
- All-purpose compute - interactive workload
- Jobs compute - pre-scheduled or submitted via API
- SQL warehouse - for sql analytics, serverless, instant startup

Autoscaling - dynamically resize cluster based on workload (setting range of number of workers require some experimenting)

Spot instances - not recommended for critical SLAs

Photon - new SQL engine (included, used when it can)

Cluster optimization Recommendations
1. DS&DE dev - app-purpose, auto-scale and auto-stop enabled, dev and test on a subset of the data
2. Ingestion & ETL jobs - jobs compute, size accordingly to job SLA
3. Ad-hoc SQL analytics - SQL warehouse, auto-scale and auto-stop enabled
4. BI reporting - isolated SQL warehouse, sized according to BI SLAs
5. Best practices:
    5.1 Enable spot instance on worker nodes
    5.2 Use the latest LTS Databricks Runtime when possible (double check working env)
    5.3 Use Photon for test TCO when applicable
    5.4 Use latest gen VM, start with general purpose, then memory/compute optimized

## Picking machines

AWS - i3's aren't always the best. Explore m7gd and r7gd
Enable caching if needed (graviton instances work well)
M7gd and r7gd have better processors
AZURE - try eav4, dav4, and f-series over l-series (ACU is very useful)
GCP - defaults very good
Usually don't need network optimized instances type some occasions they help with Python

If you 2x the cluster and it runs in 1/2 time, it cost the same!
Run set spark.sql.shuffle.partitions = 2 # of cores
Keep total memory available to the machines less than 128gb
Number of cores should be a ratio of 1 core to 128mb -> 2gb of reads
Avoid setting any other configs at first

Sizing a Driver
Size the same as worker, it do very little work in a Spark apps
4-8 core 12-32gb ram driver should be fine for most workloads
Large commits to delta tables use more memory

Set shuffle partitions to auto
spark.sql.shuffle.partitions = auto
or go to stage UI, find largest shuffle read size and divide it by 200mb.

Check event logs!